# Hidden Markov model

In [7]:
!pip install tensorflow
import tensorflow as tf

# =====================================================================
# 1. Define Model Parameters
# =====================================================================

# Initial state logits
initial_logits = tf.constant([0.6, 0.4], dtype=tf.float32)

# Convert logits to probabilities so 'initial_probabilities' is defined
initial_probabilities = tf.nn.softmax(initial_logits)

# Transition matrix (Probability of moving from state i to state j)
transition_matrix = tf.constant([
    [0.7, 0.3],
    [0.2, 0.8]
], dtype=tf.float32)

# Emission matrix (Probability of observing outcome k given state i)
emission_matrix = tf.constant([
    [0.1, 0.4, 0.5],
    [0.6, 0.3, 0.1]
], dtype=tf.float32)

# =====================================================================
# 2. Pure TF Forward Algorithm Implementation
# =====================================================================

@tf.function
def hmm_forward_log_likelihood(observed_sequence, initial_prob, trans_mat, emiss_mat):
    """
    Computes the log-likelihood of a sequence using the Forward Algorithm.
    Uses log-space arithmetic to prevent numerical underflow issues.
    """
    # 1. Map the sequence of observed IDs to their emission probabilities across all states
    # Shape: (num_steps, num_states)
    sequence_emissions = tf.gather(emiss_mat, observed_sequence, axis=1)
    sequence_emissions = tf.transpose(sequence_emissions)

    # Convert probabilities to log space to prevent underflow
    log_initial = tf.math.log(initial_prob)
    log_trans = tf.math.log(trans_mat)
    log_emissions = tf.math.log(sequence_emissions)

    # 2. Define the step function for tf.scan
    def forward_step(log_alpha_prev, log_emission_t):
        # log_alpha_prev shape: (num_states,)
        # log_emission_t shape: (num_states,)
        
        # Reshape for broadcasting matrix addition: (num_states, 1) + (num_states, num_states)
        # This sums: log_alpha_prev[i] + log_trans[i, j]
        combined = tf.expand_dims(log_alpha_prev, axis=1) + log_trans
        
        # Log-Sum-Exp over the rows (previous states) to get the prediction step
        log_alpha_predict = tf.reduce_logsumexp(combined, axis=0)
        
        # Add the emission probability for the current observation
        log_alpha_t = log_alpha_predict + log_emission_t
        return log_alpha_t

    # 3. Initialize the first step (t=0)
    log_alpha_0 = log_initial + log_emissions[0]

    # 4. Use tf.scan to loop through remaining steps (t=1 to T)
    # tf.scan handles dependencies over time smoothly inside the graph
    log_alphas = tf.scan(
        fn=forward_step,
        elems=log_emissions[1:],
        initializer=log_alpha_0
    )

    # Combine initial step with scanned steps
    all_log_alphas = tf.concat([tf.expand_dims(log_alpha_0, axis=0), log_alphas], axis=0)
    
    # 5. Final Log-Likelihood is the Log-Sum-Exp over all final states at the last step
    final_log_alpha = all_log_alphas[-1]
    total_log_likelihood = tf.reduce_logsumexp(final_log_alpha)
    
    return total_log_likelihood, all_log_alphas

# =====================================================================
# 3. Running Inference
# =====================================================================

# Observed sequence of events (Indices matching outcomes: 0, 1, or 2)
observed_sequence = tf.constant([0, 1, 2, 1, 0], dtype=tf.int32)

log_likelihood, forward_variables = hmm_forward_log_likelihood(
    observed_sequence, 
    initial_probabilities, 
    transition_matrix, 
    emission_matrix
)

print(f"\nSequence Log-Likelihood: {log_likelihood.numpy():.4f}")
print(f"Sequence Absolute Probability: {tf.exp(log_likelihood).numpy():.4f}")

# =====================================================================
# 4. Calculate Forward Probabilities (Filtered Hidden States)
# =====================================================================
# Exponentiate the log-probabilities and normalize them at each step
# to check the likely hidden state during execution
probabilities_over_time = tf.exp(forward_variables)
normalized_probs = probabilities_over_time / tf.reduce_sum(probabilities_over_time, axis=1, keepdims=True)

print("\nForward (filtered) probabilities of hidden states at each step:")
for step in range(observed_sequence.shape[0]):
    prob_state_0 = normalized_probs[step, 0].numpy()
    prob_state_1 = normalized_probs[step, 1].numpy()
    print(f"Step {step} (Observed {observed_sequence[step].numpy()}): "
          f"State 0 Prob = {prob_state_0:.2%}, State 1 Prob = {prob_state_1:.2%}")


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip

Sequence Log-Likelihood: -5.7357
Sequence Absolute Probability: 0.0032

Forward (filtered) probabilities of hidden states at each step:
Step 0 (Observed 0): State 0 Prob = 16.91%, State 1 Prob = 83.09%
Step 1 (Observed 1): State 0 Prob = 34.66%, State 1 Prob = 65.34%
Step 2 (Observed 2): State 0 Prob = 74.86%, State 1 Prob = 25.14%
Step 3 (Observed 1): State 0 Prob = 64.27%, State 1 Prob = 35.73%
Step 4 (Observed 0): State 0 Prob = 15.36%, State 1 Prob = 84.64%
